# Databricks AI Playground:

**Overview**
* **Databricks AI Playground** is an interactive, browser-based workspace (**AI/ML > Playground**) designed for rapid LLM experimentation, prompt engineering, side-by-side model benchmarking, and no-code agent prototyping.
* Every visual setting, tool attachment, and prompt rule configured in the UI maps directly to production Databricks primitives (**Foundation Model APIs**, **Unity Catalog**, **AI Search** (formerly Vector Search), **MCP Servers**, **Databricks Apps**, and **MLflow**).

## Part 1: System Prompt & User Prompt

![part1_1795114737945.png](./part1_1795114737945.png "part1_1795114737945.png")

### How AI Playground Structures Conversations
When you interact with an LLM in AI Playground, every request is sent as a structured list of role-based messages (`system`, `user`, `assistant`, and `tool`). Separating instructions into **System** and **User** prompts is the foundation of reliable prompt engineering.

### 1. System Prompt (`+ Add system prompt`)
* **Where to find it:** Directly above the bottom message box, click **`+ Add system prompt`** to expand a persistent instruction block.
* **What it does:** Defines the global rules, persona, boundaries, and formatting expectations that remain active across **every turn** of the conversation.
* **Key Presentation Talking Points:**
  * **Role & Persona:** e.g., *"You are a senior Databricks Data Engineer specializing in Spark optimization."*
  * **Output Constraints:** Enforce structured outputs such as strict JSON, Markdown tables, or bullet limits.
  * **Tool-Calling Policies:** Instruct the model *when* to use attached tools vs. when to answer directly, or tell it to refuse answering if retrieved context lacks evidence (reducing hallucinations).
  * **Safety & Guardrails:** Prevent prompt injection or off-topic responses.

### 2. User Prompt (`Start typing ...`)
* **Where to find it:** The main chat input bar at the bottom of the screen.
* **What it does:** Represents the immediate turn-by-turn query, raw text, code snippet, or task submitted by the end user.
* **Sample AI Instructions:** On a blank Playground canvas, Databricks lists one-click **sample AI instructions** you can select to get started instantly (per the official docs).

## Part 2: Settings Icon (`⚙️`) — Generation & Inference Parameters

![part2_1795114814162.png](./part2_1795114814162.png "part2_1795114814162.png")

Clicking the **gear icon (`⚙️`)** in the top-left toolbar opens the inference configuration drawer. These parameters mathematically control how the model samples the next token at every step of generation.

### Detailed Breakdown of Every Control

| Parameter | UI Control & Range | Technical Mechanism | When to Use What |
| :--- | :--- | :--- | :--- |
| **Temperature** | Toggle + Slider (`0.0` to `2.0`) | Scales the logits before applying softmax. Lower values sharpen the probability distribution toward the single most likely token; higher values flatten it. | **`0.0 – 0.2`:** SQL generation, JSON extraction, tool calling, factual Q&A.<br>**`0.7 – 1.0+`:** Brainstorming, creative writing, varied phrasing. |
| **Top P** *(Nucleus Sampling)* | Toggle + Slider (`0.0` to `1.0`, default `0.95`) | Dynamically selects the smallest set of top tokens whose cumulative probability is $\ge P$, then samples only from that subset. | Lower to `0.5 – 0.8` to eliminate bizarre "long-tail" word choices while keeping natural phrasing. |
| **Top K** | Toggle + Slider (`1` to `100+`) | Hard-caps the candidate pool at each token step to strictly the top $K$ highest-probability tokens, regardless of probability mass. | Set `Top K = 1` for greedy decoding; use `20 – 50` to bound vocabulary diversity on supported models. |
| **Max tokens** | Numeric box (e.g., `5000`) | Sets the maximum number of output tokens the model is permitted to generate in a single response before stopping. | Increase for long code generation or detailed reports; lower to control latency and DBU cost. |
| **Add other parameter** | Searchable dropdown (`Add parameter`) | Allows injecting any additional endpoint-supported OpenAI/Databricks request parameter by typing its name and pressing **Enter**. | Pass parameters like `stop` (stop sequences), `frequency_penalty`, `presence_penalty`, `seed`, or `reasoning_effort`. |

## Part 3: Model Selection Dropdown (`Select an endpoint`)

![part3_1795114896155.png](./part3_1795114896155.png "part3_1795114896155.png")

Clicking the active model name at the top center (e.g., `GPT OSS 120B`) opens the **Select an endpoint** browser, which unifies all LLMs accessible in your workspace under one governed interface.

### What You See in the Model Selector
1. **Search & Model Type Filter:** Filter endpoints by name or endpoint category.
2. **Endpoint Categories:**
   * **Databricks Foundation Model APIs (`system.ai.*`):** Pay-per-token open-weight models hosted and optimized directly by Databricks (e.g., *GPT OSS 120B*, *GPT OSS 20B*, *Qwen3 Next Instruct*, *Qwen3.5 122B A10B*, *Gemma 3 12B*, *Meta Llama 3.1 8B*, *Meta Llama 3.3 70B*, *Llama 4 Maverick*).
   * **External Models:** Third-party provider models (OpenAI, Anthropic Claude, Google Gemini, AWS Bedrock, Azure OpenAI) served through governed endpoints — **Unity Gateway** (formerly Mosaic AI Gateway) model services (`system.ai.*`) or your own external-model serving endpoints — with centralized credential management, rate limiting, and payload logging.
   * **Custom / Provisioned Throughput Endpoints:** Your own fine-tuned models or private weights registered in Unity Catalog and deployed on Databricks Model Serving.
3. **Right-Hand Inspector Card (Critical for Architecture Decisions):**
   * **Status Badge:** Confirms the endpoint is `Ready`.
   * **`Tools enabled` Badge:** Indicates whether the model supports native function/tool calling (required to unlock the `Tools` dropdown).
   * **Architecture & Context Window:** Displays model capabilities (e.g., chain-of-thought reasoning, `128K token context window`).
   * **Transparent Cost Metrics (`DBUs per 1M tokens`):** Shows exact **Input DBUs** vs. **Output DBUs** (e.g., `2.143 DBUs` input / `8.571 DBUs` output) so teams can evaluate cost-performance trade-offs before going to production.

## Part 4: Multi-Model Comparison (`+` Button)

![part4_1795115033572.png](./part4_1795115033572.png "part4_1795115033572.png")

In the top-right corner of AI Playground, clicking the **`+` (Add endpoint)** icon transforms the single-chat view into a **side-by-side benchmarking arena**.

### How Multi-Model Comparison Works
* **Parallel Panes (Multiple Models):** Add additional comparison columns—for example, comparing a large reasoning model (`GPT OSS 120B`), a fast mid-size model (`Meta Llama 3.3 70B`), and an external model (`Gemini` or `Claude`) on the exact same task.
* **Per-Pane Independence:** Each column maintains its own model selection, parameter settings (`⚙️`), and tool configuration—meaning you can even compare **the same model with two different Temperatures or System Prompts** (A/B prompt testing).
* **Synchronized Prompting:** Select the per-pane **Sync** checkboxes to synchronize queries — a new prompt is then dispatched to all synced panes simultaneously.
* **Built-In Performance & Cost Telemetry:** Under every generated answer, AI Playground reports:
  * **Time to First Token (TTFT):** How fast streaming begins (critical for user-facing chat UX).
  * **Total Generation Time & Tokens/sec:** End-to-end throughput speed.
  * **Input & Output Token Counts:** Directly maps to the DBU pricing shown in the model selector.

## Part 5: Tool Selection — Tab 1: UC Function (Unity Catalog Functions)

![part5_tab1_1795115146796.png](./part5_tab1_1795115146796.png "part5_tab1_1795115146796.png")

Clicking **`Tools > + Add tool`** opens the **Add tools** modal. The first tab, **UC Function**, connects your LLM to **real, executable SQL or Python functions** governed by Unity Catalog.

### How UC Functions Work as Agent Tools
* **What you can select:**
  * **Pre-built Databricks System Tools (`system.ai.*`):** Out-of-the-box utilities like `system.ai.python_exec` (sandboxed Python code interpreter) or `system.ai.ai_analyze_sentiment`.
  * **Custom Unity Catalog Functions (`catalog.schema.function_name`):** Any SQL or Python UDF your data team created in Unity Catalog (e.g., `claims.genai.lookup_claim`, `claims.genai.search_policy`).
  * **Wildcard Schema Selection (`catalog.schema.*`):** Attach all functions inside a schema at once.
* **Execution Lifecycle:**
  1. AI Playground automatically converts the UC function's parameter types and `COMMENT` metadata into a JSON tool schema for the LLM.
  2. When a user asks a question requiring live data or computation, the LLM emits a structured tool call with extracted arguments.
  3. **Databricks executes the function automatically** using your Unity Catalog permissions and feeds the return value back to the LLM to synthesize the final answer.


## Part 6: Tool Selection — Tab 2: Function Definition (Custom JSON Schema)

![part5_tab2_1795115323790.png](./part5_tab2_1795115323790.png "part5_tab2_1795115323790.png")

The second tab in the **Add tools** modal, **Function Definition**, lets you define an **OpenAI-compatible JSON function specification** directly inside the browser without creating anything in Unity Catalog first.

### How Function Definition Works
* **What you configure:** A JSON object specifying `name`, `description`, `parameters` (JSON Schema `type`, `properties`, per-argument `description's`), and `required` fields (e.g., `get_distance_between_locations` requiring `location_a` and `location_b`).
* **Built-in `Examples` Dropdown:** Provides ready-made JSON schemas to test function calling immediately.
* **Crucial Architectural Distinction (vs. UC Function):**
  * **UC Function:** The LLM calls the tool **AND** Databricks executes it live.
  * **Function Definition:** The LLM **calls (generates the structured tool call & JSON arguments)**, **but does NOT execute it** in Playground.
* **Why this feature exists:** Ideal for **schema prototyping and client-side tool testing**—validating whether a model accurately extracts parameters and formats tool calls for an external microservice or frontend callback before you write backend execution code.

## Part 7: Tool Selection — Tab 3: AI Search (formerly Vector Search)

![part5_tab3_1795115387018.png](./part5_tab3_1795115387018.png "part5_tab3_1795115387018.png")

The third tab, **AI Search** (formerly Mosaic AI Vector Search), connects your agent to **AI Search indexes** (e.g., `dev.bronze.resume_index1`, `claims.genai.policy_chunks_idx`) to build **Agentic Retrieval-Augmented Generation (RAG)** in seconds. Per the official docs, when an agent uses an AI Search index, its response **cites the sources used**.

### How AI Search Works in AI Playground
* **Zero-Code RAG Wiring:** Simply select one or more synced AI Search indexes from Unity Catalog—no retriever glue code is required during prototyping (the index and its embedding pipeline are already set up).
* **Agentic Retrieval Loop:**
  1. Unlike static RAG (which blindly searches on every message, even `"Hi"`), exposing AI Search as a **tool** lets the LLM decide *when* retrieval is needed and formulate optimized search queries.
  2. The tool queries the index using hybrid/semantic similarity search, returns top matching document chunks + metadata, and the LLM grounds its response in those retrieved passages.
* **Trace Inspection:** In Playground, you can expand the tool call step in the chat UI to inspect the exact search query the LLM generated and the raw document chunks returned by the index.

## Part 8: Tool Selection — Tab 4: MCP Servers (Model Context Protocol)

![part5_tab4_1795115484882.png](./part5_tab4_1795115484882.png "part5_tab4_1795115484882.png")

The fourth tab, **MCP Servers**, integrates the open **Model Context Protocol (MCP)** standard directly into Databricks—allowing agents to discover and invoke governed internal capabilities or external services through a unified protocol.

### Breakdown of All 6 MCP Server Sections in AI Playground

1. **UC MCP Service (`MCP Server`)**
   * Connects to Unity Catalog–governed MCP services (such as Databricks-managed system MCP services).
   * **Key Architecture Detail:** Requests are routed through **Unity Gateway**, enforcing centralized Unity Catalog permissions, rate limits, guardrails, and audit logging.
2. **Unity Catalog Function (`MCP Server`)**
   * Select one or more Unity Catalog schemas (`Browse`) to expose their SQL/Python functions as an MCP tool server.
3. **AI Search (`MCP Server`)**
   * Select one or more Unity Catalog schemas (`Browse`) to expose their AI Search indexes (formerly Vector Search) over MCP for unstructured document retrieval.
4. **Genie Agent (`MCP Server`)**
   * Connect one or more **Genie Agents** (formerly Genie spaces) so your orchestrator agent can ask natural-language analytical questions against curated tables and metric views—letting Genie generate and validate the SQL behind the scenes.
5. **Custom MCP Server (`MCP Server`)**
   * Connect custom MCP servers that you built and deployed on **Databricks Apps**.
   * **Naming Convention:** Only Databricks Apps whose names start with **`mcp-`** appear in this dropdown.
6. **UC Connection (Legacy) (`MCP Server`)**
   * Connects external HTTP/MCP services defined via **Unity Catalog Connections** (marked *Legacy* as newer architectures standardize around Unity Gateway and `UC MCP Service`).

## Part 9: `Get code` Dropdown — Moving from Prototype to Production

![part9_1795115666527.png](./part9_1795115666527.png "part9_1795115666527.png")

Once you have validated your prompt, parameters, and tools in AI Playground, clicking **`Get code`** in the top toolbar bridges the gap between experimentation and engineering with **documented export paths and code snippets**:

| Option | Badge / Status | What It Generates | Best Production Use Case |
| :--- | :--- | :--- | :--- |
| **1. Export to Databricks Apps** | **`Recommended`** | Installs a deployable **Databricks App** from the `agent-openai-agents-sdk` template (name must start with `agent-`), bundling your agent server, built-in chat web UI, MCP tool wiring, authentication, MLflow tracing, and automatic resource bindings. | Sharing a governed web app with business SMEs or deploying a full-stack agent quickly. |
| **2. Create agent notebook** | **`Legacy`** | Generates a Python notebook whose driver defines a tool-calling MLflow `ResponsesAgent`, tests it locally, uses MLflow code-based logging, registers the agent in Unity Catalog, and deploys it to Model Serving. | Data scientists/engineers who want full custom Python control. Databricks recommends Databricks Apps for new agents. |
| **3. Curl API** | REST Snippet | Copies a ready-to-run `curl` command with your workspace endpoint URL, auth header placeholder, messages array, and inference parameters (with tools attached and a Supervisor-compatible model, it is a Supervisor API POST to `/mlflow/v1/responses`). | Testing endpoints from terminal/shell scripts or integrating with non-Python services. |
| **4. Python API** | SDK Snippet | Copies a clean Python script using `DatabricksOpenAI` / `OpenAI` client configured with your exact model endpoint, prompts, and settings. | Embedding the LLM call directly inside existing Python applications, microservices, or notebooks. |
| **5. Apply on data** | Batch Snippet | Generates a SQL query / notebook that batch-scores your validated prompt and model over an **entire Unity Catalog table** — typically via `ai_query()` against the same serving endpoint, applying your configuration as a labeling or enrichment task to every row. | Batch enrichment, classification, or extraction at scale (e.g., tagging support tickets, summarizing product reviews, extracting entities) where interactive chat is impractical. |

> **Note:** Option **5 (`Apply on data`)** is the batch path — it leverages the same `ai_query` capability documented under AI Functions, but exports it directly from your validated Playground configuration so your tested prompt is applied consistently at table scale.


## Part 10: Live Code Execution Lab

Example code to show how **Python API** and **Multi-Model Comparison (`+`)** features work programmatically under the hood.

```bash
%pip install -U -qqqq databricks-openai openai
dbutils.library.restartPython()
```

```python
# Demo A: Replicating System Prompt, User Prompt, Model Selection & Settings in Python
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI()

response = client.chat.completions.create(
    model="databricks-gpt-oss-120b",  # Selected in Model Dropdown
    messages=[
        {
            "role": "system",
            "content": "You are a Databricks Solutions Architect. Respond in 3 concise bullet points."
        },
        {
            "role": "user",
            "content": "How does AI Playground accelerate GenAI development on Databricks?"
        }
    ],
    temperature=0.2,   # Configured in Settings (⚙️)
    top_p=0.95,        # Configured in Settings (⚙️)
    max_tokens=400,    # Configured in Settings (⚙️)
)

print(response.choices[0].message.content)
```

```python
# Demo B: Programmatic Multi-Model Comparison (Simulating the '+' Button)
import time

models = [
    "databricks-gpt-oss-120b",
    "databricks-meta-llama-3-3-70b-instruct",
]

test_prompt = "Explain Unity Catalog in one clear sentence for an executive audience."

for endpoint in models:
    t0 = time.time()
    res = client.chat.completions.create(
        model=endpoint,
        messages=[{"role": "user", "content": test_prompt}],
        temperature=0.0,
        max_tokens=120,
    )
    latency = round(time.time() - t0, 2)
    usage = res.usage
    print(f"Model: {endpoint}")
    print(f"Latency: {latency}s | Prompt Tokens: {usage.prompt_tokens} | Completion Tokens: {usage.completion_tokens}")
    print(f"Response: {res.choices[0].message.content}")
    print("-" * 80)
```
